# Gated Attention for Vision Transformers


## 0. Setup

In [ ]:
import os, types, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.stats import pearsonr
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
import timm
from timm.layers import Attention

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_DIR = Path('checkpoints_B')
CKPT_DIR.mkdir(exist_ok=True)
print(f'torch {torch.__version__} | timm {timm.__version__} | device {DEVICE}')
print(f'Checkpoints → {CKPT_DIR.resolve()}')

## 1. Dataset Paths & Loaders

In [ ]:
IN1K_TRAIN = Path('/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train')
IN1K_VAL   = Path('/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val')

ADE_ROOT     = Path('/kaggle/input/datasets/ipythonx/ade20k-scene-parsing/ADEChallengeData2016')
ADE_TRAIN_IMG = ADE_ROOT / 'images/training'
ADE_VAL_IMG   = ADE_ROOT / 'images/validation'
ADE_TRAIN_ANN = ADE_ROOT / 'annotations/training'
ADE_VAL_ANN   = ADE_ROOT / 'annotations/validation'

for p in [IN1K_TRAIN, IN1K_VAL, ADE_VAL_IMG, ADE_VAL_ANN]:
    print(f'  {p.name:<30} {"Yes" if p.exists() else "MISSING"}')

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
BS       = 64
SLICE    = 0.1
VIT_SIZE = 224
MASK_SIZE = 512

train_tfm = T.Compose([
    T.RandomResizedCrop(224), T.RandomHorizontalFlip(),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
val_tfm = T.Compose([
    T.Resize(256), T.CenterCrop(224),
    T.ToTensor(), T.Normalize(MEAN, STD),
])

def build_imagenet_samples(root, frac):
    class_dirs   = sorted(p for p in Path(root).iterdir() if p.is_dir())
    class_to_idx = {d.name: i for i, d in enumerate(class_dirs)}
    samples = []
    for d in class_dirs:
        files = sorted(d.glob('*.JPEG')) + sorted(d.glob('*.jpg'))
        keep  = max(1, int(len(files) * frac))
        for f in files[:keep]:
            samples.append((str(f), class_to_idx[d.name]))
    return samples

print('Scanning ImageNet-1K train ...')
all_samples = build_imagenet_samples(IN1K_TRAIN, SLICE)
random.shuffle(all_samples)
split       = int(len(all_samples) * 0.9)
train_samp  = all_samples[:split]
val_samp    = all_samples[split:]

class ImageSamples(Dataset):
    def __init__(self, samples, tfm):
        self.samples, self.tfm = samples, tfm
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        return self.tfm(Image.open(path).convert('RGB')), label

in1k_train = ImageSamples(train_samp, train_tfm)
in1k_val   = ImageSamples(val_samp,   val_tfm)
in1k_train_loader = DataLoader(in1k_train, BS, shuffle=True,  num_workers=4, pin_memory=True)
in1k_val_loader   = DataLoader(in1k_val,   BS, shuffle=False, num_workers=4, pin_memory=True)
print(f'ImageNet-1K  train {len(in1k_train):,} | val {len(in1k_val):,}  ({SLICE*100:.0f}% slice, 90/10 split)')

class ADE20KDataset(Dataset):
    def __init__(self, img_dir, ann_dir, frac=1.0):
        imgs = sorted(img_dir.glob('*.jpg'))
        anns = sorted(ann_dir.glob('*.png'))
        n = max(1, int(len(imgs) * frac))
        self.imgs, self.anns = imgs[:n], anns[:n]
        self.img_tfm = T.Compose([
            T.Resize((VIT_SIZE, VIT_SIZE)),
            T.ToTensor(), T.Normalize(MEAN, STD),
        ])

    def __len__(self): return len(self.imgs)

    def __getitem__(self, i):
        img  = self.img_tfm(Image.open(self.imgs[i]).convert('RGB'))
        mask = np.array(Image.open(self.anns[i]).resize(
            (MASK_SIZE, MASK_SIZE), Image.NEAREST), dtype=np.int64)
        mask = torch.from_numpy(mask).long() - 1
        return img, mask

ade_train = ADE20KDataset(ADE_TRAIN_IMG, ADE_TRAIN_ANN, frac=SLICE)
ade_val   = ADE20KDataset(ADE_VAL_IMG,   ADE_VAL_ANN,   frac=SLICE)
ade_train_loader = DataLoader(ade_train, 16, shuffle=True,  num_workers=4, pin_memory=True)
ade_val_loader   = DataLoader(ade_val,   16, shuffle=False, num_workers=4, pin_memory=True)
print(f'ADE20K  train {len(ade_train):,} | val {len(ade_val):,}  ({SLICE*100:.0f}% slice)')
print(f'ViT input: {VIT_SIZE}×{VIT_SIZE}  |  Mask size: {MASK_SIZE}×{MASK_SIZE}')

## 2. Gate Modules — G1 through G5 + PNG

In [ ]:
class GateParams(nn.Module):
    """Learnable parameters for one attention layer's gate."""
    def __init__(self, dim, num_heads, png=False):
        super().__init__()
        self.W   = nn.Parameter(torch.empty(dim, num_heads).normal_(std=0.02))
        self.png = png
        if png:

            self.e    = nn.Parameter(torch.ones(num_heads))
            self.beta = nn.Parameter(torch.zeros(1))

    def gate(self, x):

        g = x @ self.W
        if self.png:
            g = g - self.beta * (x.norm(dim=-1, keepdim=True) + 1e-6) * self.e
        return torch.sigmoid(g)

def _patched_forward(mod, params, pos):
    """Single forward that covers all 5 gate positions (and PNG = G1 variant)."""
    def forward(self, x, attn_mask=None):
        B, N, C = x.shape
        H, D = self.num_heads, self.head_dim

        qkv = self.qkv(x).reshape(B, N, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q, k = self.q_norm(q), self.k_norm(k)

        if pos == 'G3':
            k = k * params.gate(x).permute(0,2,1).unsqueeze(-1)
        if pos == 'G4':
            q = q * params.gate(x).permute(0,2,1).unsqueeze(-1)
        if pos == 'G2':
            v = v * params.gate(x).permute(0,2,1).unsqueeze(-1)

        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask,
            dropout_p=self.attn_drop.p if self.training else 0.)

        if pos in ('G1', 'PNG'):
            G = params.gate(x)
            out = (out.transpose(1,2) * G.unsqueeze(-1)).transpose(1,2)
            if getattr(self, '_capture', False):
                self._last_gate   = G.detach()
                self._last_x_norm = (x.norm(dim=-1, keepdim=True) + 1e-6).detach()

        x = out.transpose(1,2).reshape(B, N, C)

        if pos == 'G5':
            x = (x.view(B,N,H,D) * params.gate(x).unsqueeze(-1)).view(B,N,C)

        x = self.proj(x)
        x = self.proj_drop(x)
        return x

    return types.MethodType(forward, mod)

def inject_gates(model, pos='G1', png=False):
    gate_list = []
    p_label = 'PNG' if png else pos
    for _, mod in model.named_modules():
        if isinstance(mod, Attention):
            p = GateParams(mod.qkv.in_features, mod.num_heads, png=png)
            mod.forward = _patched_forward(mod, p, p_label)
            gate_list.append(p)
    model.gate_params = nn.ModuleList(gate_list)
    return model

print('Gate code ready — covers G1/G2/G3/G4/G5/PNG')

## 3. Classification Helpers (ImageNet-1K)

In [ ]:
def build_vit(pos='baseline', png=False, num_classes=1000, pretrained=True):
    """Build ViT. Pass pretrained=False when loading from a local checkpoint
    to avoid an unnecessary (and potentially failing) network download."""
    m = timm.create_model('vit_base_patch16_224', pretrained=pretrained, num_classes=num_classes)
    if pos != 'baseline':
        m = inject_gates(m, pos=pos, png=png)
    for p in m.parameters():             p.requires_grad_(False)
    for p in m.head.parameters():        p.requires_grad_(True)
    if hasattr(m, 'gate_params'):
        for p in m.gate_params.parameters(): p.requires_grad_(True)
    return m.to(DEVICE)

@torch.no_grad()
def evaluate_cls(model, loader, num_classes=1000):
    """Returns acc, macro-F1, macro-precision, macro-recall."""
    model.eval()
    all_preds, all_labels = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        preds = model(x).argmax(1)
        all_preds.append(preds.cpu())
        all_labels.append(y.cpu())
    preds  = torch.cat(all_preds)
    labels = torch.cat(all_labels)

    acc = (preds == labels).float().mean().item()

    tp = torch.zeros(num_classes)
    fp = torch.zeros(num_classes)
    fn = torch.zeros(num_classes)
    for c in range(num_classes):
        pred_c = preds  == c
        true_c = labels == c
        tp[c]  = (pred_c & true_c).sum()
        fp[c]  = (pred_c & ~true_c).sum()
        fn[c]  = (~pred_c & true_c).sum()

    present   = (tp + fn) > 0
    prec_c    = tp / (tp + fp + 1e-8)
    rec_c     = tp / (tp + fn + 1e-8)
    f1_c      = 2 * prec_c * rec_c / (prec_c + rec_c + 1e-8)
    precision = prec_c[present].mean().item()
    recall    = rec_c[present].mean().item()
    f1        = f1_c[present].mean().item()
    return acc, f1, precision, recall

@torch.no_grad()
def top1(model, loader):
    acc, _, _, _ = evaluate_cls(model, loader)
    return acc

def train_cls(model, epochs=5, lr=1e-3, ckpt_path=None):
    """Train classification head (+ optional gate params).

    ckpt_path: if given, the model state is saved after every epoch so that
    training can be resumed without losing progress on an error.
    """
    crit = nn.CrossEntropyLoss()
    opt  = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    log = {'loss': [], 'acc': [], 'f1': [], 'precision': [], 'recall': []}

    for ep in range(1, epochs+1):
        model.train()
        for name, mod in model.named_modules():
            if 'gate_params' not in name and 'head' not in name \
                    and isinstance(mod, nn.LayerNorm):
                mod.eval()
        total_loss = 0
        for x, y in tqdm(in1k_train_loader, desc=f'ep{ep}', leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
            total_loss += loss.item()
        avg = total_loss / len(in1k_train_loader)
        acc, f1, prec, rec = evaluate_cls(model, in1k_val_loader)
        sched.step()
        log['loss'].append(avg)
        log['acc'].append(acc)
        log['f1'].append(f1)
        log['precision'].append(prec)
        log['recall'].append(rec)
        print(f'  ep{ep}  loss={avg:.4f}  acc={acc:.4f}  '
              f'f1={f1:.4f}  prec={prec:.4f}  rec={rec:.4f}')

        if ckpt_path is not None:
            torch.save(model.state_dict(), ckpt_path)

    return log

print('Classification helpers ready.')

## 4. Run G1–G5 + PNG Ablation (ImageNet-1K)

In [ ]:
EPOCHS_CLS = 2

CKPT_NAMES = {
    'Baseline':          'vit_baseline.pt',
    'G1 — SDPA output':  'vit_g1.pt',
    'G2 — Value proj':   'vit_g2.pt',
    'G3 — Key proj':     'vit_g3.pt',
    'G4 — Query proj':   'vit_g4.pt',
    'G5 — Dense output': 'vit_g5.pt',
    'PNG (novel)':        'vit_png.pt',
}

experiments = [
    ('Baseline',          'baseline', False),
    ('G1 — SDPA output',  'G1',       False),
    ('G2 — Value proj',   'G2',       False),
    ('G3 — Key proj',     'G3',       False),
    ('G4 — Query proj',   'G4',       False),
    ('G5 — Dense output', 'G5',       False),
    ('PNG (novel)',        'G1',       True),
]

cls_results = {}
for name, pos, png in experiments:
    print(f'\n── {name} ──')
    ckpt_path = CKPT_DIR / CKPT_NAMES[name]
    if ckpt_path.exists():

        print(f'  Loading checkpoint: {ckpt_path}')
        model = build_vit(pos, png, pretrained=False)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        acc, f1, prec, rec = evaluate_cls(model, in1k_val_loader)
        log = {'loss': [], 'acc': [acc], 'f1': [f1], 'precision': [prec], 'recall': [rec]}
        print(f'  Loaded — acc={acc:.4f}  f1={f1:.4f}  prec={prec:.4f}  rec={rec:.4f}')
    else:

        model = build_vit(pos, png, pretrained=True)
        n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in model.parameters())
        print(f'  trainable {n_train:,} / {n_total:,}')
        log = train_cls(model, epochs=EPOCHS_CLS, ckpt_path=ckpt_path)
        print(f'  Final checkpoint saved → {ckpt_path}')
    cls_results[name] = {'model': model, 'log': log}

## 5. Segmentation Head — Linear Probe on ADE20K

In [ ]:
ADE_CLASSES = 150
PATCH_GRID  = VIT_SIZE // 16

class LinearSegHead(nn.Module):
    """
    Takes features from the last 4 ViT blocks (each 14×14 for 224px input),
    concatenates at 14×14, applies 1×1 conv to predict per-pixel class,
    then upsamples to MASK_SIZE.

    get_intermediate_layers(reshape=False) already strips prefix tokens
    (CLS etc.) and returns only patch tokens: (B, 196, 768).
    No manual slicing needed.
    """
    def __init__(self, embed_dim=768, num_classes=150):
        super().__init__()
        self.head = nn.Conv2d(embed_dim * 4, num_classes, kernel_size=1)

    def forward(self, feats, out_size):

        maps = []
        for f in feats:
            B, N, C = f.shape
            f = f.transpose(1, 2).reshape(B, C, PATCH_GRID, PATCH_GRID)
            maps.append(f)
        x = self.head(torch.cat(maps, dim=1))
        return F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)

class SegModel(nn.Module):
    def __init__(self, vit, seg_head):
        super().__init__()
        self.vit      = vit
        self.seg_head = seg_head

    def forward(self, x):

        feats = self.vit.get_intermediate_layers(x, n=[8, 9, 10, 11], reshape=False)

        return self.seg_head(feats, (MASK_SIZE, MASK_SIZE))

@torch.no_grad()
def mean_iou(model, loader):
    model.eval()
    inter = torch.zeros(ADE_CLASSES)
    union = torch.zeros(ADE_CLASSES)
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs).argmax(1)
        valid = masks >= 0
        for c in range(ADE_CLASSES):
            pred_c = (preds == c) & valid
            true_c = (masks == c) & valid
            inter[c] += (pred_c & true_c).sum().cpu()
            union[c] += (pred_c | true_c).sum().cpu()
    iou = inter / (union + 1e-6)
    return iou[union > 0].mean().item()

def train_seg(vit_model, epochs=10, lr=1e-4, freeze_vit=True):
    seg_head = LinearSegHead(embed_dim=768, num_classes=ADE_CLASSES).to(DEVICE)
    model = SegModel(vit_model, seg_head).to(DEVICE)

    if freeze_vit:
        for p in model.vit.parameters(): p.requires_grad_(False)
        if hasattr(model.vit, 'gate_params'):
            for p in model.vit.gate_params.parameters(): p.requires_grad_(True)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  trainable {trainable:,}')

    opt   = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                               lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss(ignore_index=-1)

    log = {'loss': [], 'miou': []}
    for ep in range(1, epochs+1):
        model.train()
        if freeze_vit: model.vit.eval()
        total = 0
        for imgs, masks in tqdm(ade_train_loader, desc=f'seg ep{ep}', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(imgs), masks)
            loss.backward()
            opt.step()
            total += loss.item()
        miou = mean_iou(model, ade_val_loader)
        sched.step()
        log['loss'].append(total / len(ade_train_loader))
        log['miou'].append(miou)
        print(f'  seg ep{ep}  loss={log["loss"][-1]:.4f}  mIoU={miou:.4f}')
    return model, log

print(f'Segmentation head ready. Patch grid {PATCH_GRID}×{PATCH_GRID} → upsample to {MASK_SIZE}×{MASK_SIZE}')

## 6. Run Segmentation — Baseline / G1 / PNG (ADE20K)

In [ ]:
EPOCHS_SEG = 2

seg_configs = [
    ('Baseline', 'Baseline'),
    ('G1 Gate',  'G1 — SDPA output'),
    ('PNG Gate', 'PNG (novel)'),
]

seg_results = {}
for seg_name, cls_key in seg_configs:
    print(f'\n── Seg: {seg_name} ──')
    vit = cls_results[cls_key]['model']
    seg_model, log = train_seg(vit, epochs=EPOCHS_SEG)
    seg_results[seg_name] = {'model': seg_model, 'log': log}

## 7. Figures

### Fig A — Classification Ablation Bar Chart

In [ ]:
names     = list(cls_results.keys())
accs      = [max(cls_results[n]['log']['acc'])       * 100 for n in names]
f1s       = [max(cls_results[n]['log']['f1'])        * 100 for n in names]
precs     = [max(cls_results[n]['log']['precision']) * 100 for n in names]
recs      = [max(cls_results[n]['log']['recall'])    * 100 for n in names]

x      = np.arange(len(names))
width  = 0.2
metric_groups = [
    (accs,  'Accuracy',  '
    (precs, 'Precision', '
    (recs,  'Recall',    '
    (f1s,   'F1',        '
]

fig, ax = plt.subplots(figsize=(13, 5))
for i, (vals, label, color) in enumerate(metric_groups):
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, vals, width, label=label, color=color,
                  edgecolor='k', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{v:.1f}', ha='center', va='bottom', fontsize=6, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=18, ha='right')
ax.set_ylabel('Score (%) — ImageNet-1K val')
ax.set_title('Gate Position Ablation: Acc / Precision / Recall / F1\n'
             '(vit_base_patch16_224, IN-21K init)',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('figA_cls_ablation.pdf', bbox_inches='tight')
plt.show()
print('Saved figA_cls_ablation.pdf')

### Fig B — Training Curves (Baseline / G1 / PNG)

In [ ]:
show    = ['Baseline', 'G1 — SDPA output', 'PNG (novel)']
palette = {'Baseline': '
eps     = range(1, EPOCHS_CLS+1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
metric_keys   = ['acc',       'f1',  'precision', 'recall']
metric_labels = ['Top-1 Acc', 'F1',  'Precision', 'Recall']

for ax, key, lbl in zip(axes, metric_keys, metric_labels):
    for n in show:
        log = cls_results[n]['log']
        vals = [v * 100 for v in log[key]]

        if len(vals) < len(eps):
            vals = vals * len(eps)
        ax.plot(list(eps)[:len(vals)], vals, 'o-', color=palette[n], label=n)
    ax.set_title(lbl); ax.set_xlabel('Epoch'); ax.set_ylabel('Score (%)')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

fig.suptitle('Training Dynamics — ImageNet-1K', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('figB_cls_curves.pdf', bbox_inches='tight')
plt.show()
print('Saved figB_cls_curves.pdf')

### Fig C — ADE20K mIoU Comparison

In [ ]:
seg_names = list(seg_results.keys())
mious     = [max(seg_results[n]['log']['miou']) * 100 for n in seg_names]
colors_s  = ['

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

bars = ax1.bar(seg_names, mious, color=colors_s, edgecolor='k', linewidth=0.6)
for bar, m in zip(bars, mious):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{m:.2f}', ha='center', va='bottom', fontsize=9)
ax1.set_ylabel('mIoU (%) — ADE20K val')
ax1.set_title('Segmentation: mIoU Comparison', fontweight='bold')
ax1.set_ylim(min(mious)*0.95, max(mious)*1.04)
ax1.grid(axis='y', alpha=0.3)

eps_seg = range(1, EPOCHS_SEG+1)
for n, c in zip(seg_names, colors_s):
    ax2.plot(eps_seg, [v*100 for v in seg_results[n]['log']['miou']], 'o-', color=c, label=n)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('mIoU (%)')
ax2.set_title('Segmentation Training Curves', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle(f'ADE20K Segmentation (Linear Probe, ViT@{VIT_SIZE}→mask@{MASK_SIZE})',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('figC_seg_miou.pdf', bbox_inches='tight')
plt.show()
print('Saved figC_seg_miou.pdf')

### Fig D — CLS Attention Heatmaps (Baseline vs G1 vs PNG)

In [ ]:
def denorm(t):
    m = torch.tensor(MEAN).view(3,1,1)
    s = torch.tensor(STD).view(3,1,1)
    return (t.cpu()*s + m).clamp(0,1).permute(1,2,0).numpy()

def get_cls_attn_map(model, batch):
    """Extract CLS-query attention from the last block via a register_forward_hook
    on the QKV linear layer — works correctly with both vanilla and gated forwards.

    Token layout inside the block: [0=CLS, 1-196=196 patch tokens]
    (this model has num_prefix_tokens=1; no dist_token)
    CLS row (index 0) to patch tokens (1:) → 196 values → 14×14 map.
    """
    attn_mod = model.blocks[-1].attn
    captured = {}

    def qkv_hook(module, inp, out):
        """Hook on attn.qkv to capture q, k after the projection."""
        x = inp[0]
        B, N, C = x.shape
        H, D = attn_mod.num_heads, attn_mod.head_dim
        qkv = out.reshape(B, N, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, _ = qkv.unbind(0)
        q, k = attn_mod.q_norm(q), attn_mod.k_norm(k)
        scale = D ** -0.5
        a = (q * scale) @ k.transpose(-2, -1)
        a = a.softmax(-1)
        captured['a'] = a.detach()

    handle = attn_mod.qkv.register_forward_hook(qkv_hook)
    model.eval()
    with torch.no_grad():
        model(batch)
    handle.remove()

    cls = captured['a'][:, :, 0, 1:].mean(1)
    return cls.reshape(-1, 14, 14).cpu().numpy()

def up_attn(a):
    a = (a - a.min()) / (a.max() - a.min() + 1e-8)
    return np.array(Image.fromarray((a*255).astype(np.uint8)).resize((224,224), Image.BILINEAR)) / 255.0

vis_imgs, _ = next(iter(DataLoader(in1k_val, batch_size=4, shuffle=False)))
vis = vis_imgs.to(DEVICE)

m_b = cls_results['Baseline']['model']
m_g = cls_results['G1 — SDPA output']['model']
m_p = cls_results['PNG (novel)']['model']

a_b = get_cls_attn_map(m_b, vis)
a_g = get_cls_attn_map(m_g, vis)
a_p = get_cls_attn_map(m_p, vis)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for j, t in enumerate(['Original', 'Baseline', 'G1 Gate', 'PNG Gate']):
    axes[0, j].set_title(t, fontsize=11, fontweight='bold')
for i in range(4):
    orig = denorm(vis_imgs[i])
    axes[i, 0].imshow(orig); axes[i, 0].axis('off')
    for j, att in enumerate([a_b[i], a_g[i], a_p[i]]):
        axes[i, j+1].imshow(orig)
        axes[i, j+1].imshow(up_attn(att), alpha=0.55, cmap='jet')
        axes[i, j+1].axis('off')
fig.suptitle('CLS Attention Maps — Final Block, Head-Averaged\n(ImageNet-1K val)',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('figD_attn_maps.pdf', bbox_inches='tight')
plt.show()
print('Saved figD_attn_maps.pdf')

### Fig E — Layer-wise Artifact Patch Attention

In [ ]:
NORM_THRESH = 150.0

def artifact_attn_per_layer(model, imgs, thresh=NORM_THRESH):
    """% of CLS attention going to artifact patches (norm > thresh) per block.

    Uses register_forward_hook on attn.qkv so the actual forward (including
    gates for G1/PNG models) runs undisturbed. We compute softmax attention
    from q, k inside the hook to capture the raw attention distribution.

    Token layout inside each block: [0=CLS, 1-196=patch tokens]
    (num_prefix_tokens=1, no dist_token for vit_base_patch16_224)
    """

    all_captured = {}
    handles = []

    for block_idx, block in enumerate(model.blocks):
        attn_mod = block.attn
        captured = {}
        all_captured[block_idx] = captured

        def make_hook(attn, cap):
            def qkv_hook(module, inp, out):
                x = inp[0]
                B, N, C = x.shape
                H, D = attn.num_heads, attn.head_dim
                qkv = out.reshape(B, N, 3, H, D).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv.unbind(0)
                q, k = attn.q_norm(q), attn.k_norm(k)
                scale = D ** -0.5
                a = (q * scale) @ k.transpose(-2, -1)
                a = a.softmax(-1)
                cap['a'] = a.detach()
                cap['x'] = x.detach()
            return qkv_hook

        h = attn_mod.qkv.register_forward_hook(make_hook(attn_mod, captured))
        handles.append(h)

    model.eval()
    with torch.no_grad():
        model(imgs.to(DEVICE))

    for h in handles:
        h.remove()

    pcts = []
    for block_idx in range(len(model.blocks)):
        cap = all_captured[block_idx]

        norms    = cap['x'].norm(dim=-1)[:, 1:]
        hi       = (norms > thresh).float()
        cls_attn = cap['a'][:, :, 0, 1:].mean(1)
        pct      = (cls_attn * hi).sum(-1) / (cls_attn.sum(-1) + 1e-8)
        pcts.append(pct.mean().item() * 100)
    return pcts

sample, _ = next(iter(DataLoader(in1k_val, batch_size=32, shuffle=False)))
print('Computing layer-wise artifact attention ...')
art_b = artifact_attn_per_layer(m_b, sample)
art_g = artifact_attn_per_layer(m_g, sample)
art_p = artifact_attn_per_layer(m_p, sample)

layers = range(1, 13)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(layers, art_b, 'o-', color='
ax.plot(layers, art_g, 's-', color='
ax.plot(layers, art_p, '^-', color='
ax.set_xlabel('Transformer Block'); ax.set_ylabel('% CLS Attention to Artifact Patches')
ax.set_title(f'Layer-wise Artifact Attention (token norm > {NORM_THRESH:.0f})\n'
              'Answers RQ1: Does G1/PNG reduce artifact patches?',
             fontsize=11, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('figE_artifact_attn.pdf', bbox_inches='tight')
plt.show()
print('Saved figE_artifact_attn.pdf')

### Fig F — PNG Gate Suppression vs Token Norm (Scatter)

In [ ]:
last_attn = m_p.blocks[-1].attn
last_attn._capture = True

all_norms, all_gates = [], []
m_p.eval()
with torch.no_grad():
    for x, _ in DataLoader(in1k_val, batch_size=64, shuffle=False):
        m_p(x.to(DEVICE))
        all_gates.append(last_attn._last_gate.mean(-1).cpu().flatten().numpy())
        all_norms.append(last_attn._last_x_norm.squeeze(-1).cpu().flatten().numpy())
        if len(all_norms) * 64 > 8000: break

last_attn._capture = False
norms = np.concatenate(all_norms)
gates = np.concatenate(all_gates)

idx = np.random.default_rng(42).choice(len(norms), 5000, replace=False)
ns, gs = norms[idx], gates[idx]
r, pv  = pearsonr(ns, gs)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(ns, gs, s=4, alpha=0.3, color='steelblue')
xl = np.linspace(ns.min(), ns.max(), 200)
m_, b_ = np.polyfit(ns, gs, 1)
ax.plot(xl, m_*xl+b_, 'r-', lw=2, label=f'fit  r={r:.3f}')
ax.set_xlabel('Token L2 Norm (pre-attention)')
ax.set_ylabel('Mean Gate Value (over heads)')
ax.set_title('PNG: Gate Suppresses High-Norm (Artifact) Tokens\nFinal Block, ImageNet-1K val',
             fontsize=11, fontweight='bold')
ax.legend()
ax.text(0.97, 0.97, f'r = {r:.3f}\np = {pv:.1e}',
        transform=ax.transAxes, ha='right', va='top',
        bbox=dict(boxstyle='round', fc='wheat', alpha=0.8))
fig.tight_layout()
fig.savefig('figF_gate_scatter.pdf', bbox_inches='tight')
plt.show()
print(f'Saved figF_gate_scatter.pdf  r={r:.3f}')

### Fig G — ADE20K Segmentation Qual Examples

In [ ]:
rng_c = np.random.default_rng(0)
CMAP  = np.vstack([[0,0,0], rng_c.integers(50, 255, (150, 3))]).astype(np.uint8)

def seg_to_rgb(mask_np):
    mask_np = np.clip(mask_np + 1, 0, 150)
    return CMAP[mask_np]

def denorm_seg(t):
    m = torch.tensor(MEAN).view(3,1,1)
    s = torch.tensor(STD).view(3,1,1)
    return (t.cpu()*s + m).clamp(0,1).permute(1,2,0).numpy()

vis_loader  = DataLoader(ade_val, batch_size=4, shuffle=False)
val_imgs, val_masks = next(iter(vis_loader))

m_seg_b = seg_results['Baseline']['model']
m_seg_g = seg_results['G1 Gate']['model']
m_seg_p = seg_results['PNG Gate']['model']

def get_pred(seg_model, imgs):
    seg_model.eval()
    with torch.no_grad():
        return seg_model(imgs.to(DEVICE)).argmax(1).cpu().numpy()

pred_b = get_pred(m_seg_b, val_imgs)
pred_g = get_pred(m_seg_g, val_imgs)
pred_p = get_pred(m_seg_p, val_imgs)

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for j, t in enumerate(['Image', 'GT Mask', 'Baseline', 'G1 Gate', 'PNG Gate']):
    axes[0, j].set_title(t, fontsize=10, fontweight='bold')

for i in range(4):
    axes[i, 0].imshow(denorm_seg(val_imgs[i])); axes[i, 0].axis('off')
    axes[i, 1].imshow(seg_to_rgb(val_masks[i].numpy())); axes[i, 1].axis('off')
    axes[i, 2].imshow(seg_to_rgb(pred_b[i])); axes[i, 2].axis('off')
    axes[i, 3].imshow(seg_to_rgb(pred_g[i])); axes[i, 3].axis('off')
    axes[i, 4].imshow(seg_to_rgb(pred_p[i])); axes[i, 4].axis('off')

fig.suptitle('ADE20K Segmentation Predictions (512×512)', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('figG_seg_qual.pdf', bbox_inches='tight')
plt.show()
print('Saved figG_seg_qual.pdf')

## 8. Summary

In [ ]:
print('=' * 75)
print('  RESULTS SUMMARY')
print('  Model: vit_base_patch16_224 (IN-21K pretrained)')
print('=' * 75)

print('\nImageNet-1K Classification (macro, val set):')
print(f'  {"Model":<25} {"Acc":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-'*62)
for name in cls_results:
    log = cls_results[name]['log']
    acc  = max(log['acc'])
    prec = max(log['precision'])
    rec  = max(log['recall'])
    f1   = max(log['f1'])
    tag  = '  ← novel' if 'PNG' in name else ''
    print(f'  {name:<25} {acc*100:>7.2f}% {prec*100:>9.2f}% {rec*100:>7.2f}% {f1*100:>7.2f}%{tag}')

print('\nADE20K Segmentation (mIoU):')
print(f'  {"Model":<20} {"Best mIoU":>10}')
print('  ' + '-'*33)
for name in seg_results:
    miou = max(seg_results[name]['log']['miou'])
    print(f'  {name:<20} {miou*100:>9.2f}%')

print('\nFigures:')
figs = [
    ('figA_cls_ablation.pdf',  'G1–G5 + PNG Acc/Prec/Rec/F1 grouped bar chart'),
    ('figB_cls_curves.pdf',    'Classification training curves (4 metrics)'),
    ('figC_seg_miou.pdf',      'ADE20K mIoU bar + training curves'),
    ('figD_attn_maps.pdf',     'CLS attention heatmaps (4 images)'),
    ('figE_artifact_attn.pdf', 'Layer-wise artifact attention (RQ1)'),
    ('figF_gate_scatter.pdf',  'PNG gate suppression vs token norm'),
    ('figG_seg_qual.pdf',      'ADE20K qualitative predictions'),
]
for f, d in figs:
    ok = '✓' if Path(f).exists() else '✗'
    print(f'  {ok} {f:<40} {d}')
print('=' * 75)